# System 1: Baseline RAG (Monolith Showcase)

This notebook demonstrates **System 1: Baseline RAG (Monolith)**. 
Unlike System 2, this is a fixed pipeline `chunking -> vectorstore -> hybrid retrieval -> reranking -> generation`.

All components are non-agentic with no autonomous decisions or tool use. It relies purely on the standard RAG process using the provided optimal configurations.


In [ ]:
import os
import sys
from pathlib import Path

# Ensure src/ is in the python path
if Path("src").exists():
    sys.path.append(os.path.abspath("."))
else:
    sys.path.append(os.path.abspath("../../"))

from dotenv import load_dotenv
load_dotenv()

from src.common.ingestion import ProcessedFiling
from src.systems.rag_monolith.pipeline import MonolithRAGPipeline

import warnings
warnings.filterwarnings('ignore')


## 1. Load Data & Build Pipeline
We load a small sample of parsed SEC 10-K filings and build the monolith pipeline.
Behind the scenes, this:
1. Chunks the documents based on config parameters.
2. Builds the ChromaDB vectorstore (or loads if cached).
3. Configures the hybrid retriever and FlashRank reranker.
4. Initializes the generation LLM.


In [ ]:
# Load parsed JSON filings from disk
# Note: Ensure you have run the ingestion pipeline first!
from pathlib import Path
import os

# Dynamically resolve data directory regardless of whether the notebook 
# is running from the project root or the showcases folder
if Path("data/processed").exists():
    data_dir = Path("data/processed")
else:
    data_dir = Path("../../data/processed")

filings = []
if data_dir.exists():
    # Find .meta.json sidecars
    for meta_file in data_dir.rglob("*.meta.json"):
        md_file = meta_file.with_suffix("").with_suffix(".md")
        if md_file.exists():
            filings.append(ProcessedFiling.from_files(md_file, meta_file))
    
print(f"Loaded {len(filings)} filings from {data_dir.absolute()}")

# Initialize pipeline
print("Building Monolith RAG Pipeline...")
pipeline = MonolithRAGPipeline()
pipeline.build(filings)
print("Pipeline built successfully!")


## 2. Basic Query
Let's ask a question about Apple. The monolith will retrieve the top documents contextually and generate an answer based purely on that context.


In [ ]:
query1 = "What are the key risk factors for Apple (AAPL) in FY2024?"
print(f"Query: {query1}\n")

res1 = pipeline.query(query1)

print("\n" + "="*50)
print(f"Answer:\n{res1.answer}")
print("="*50 + "\n")

print(f"Tokens Used: {res1.metrics.token_usage.total_tokens}")
print(f"Latency: {res1.metrics.latency_seconds:.2f}s")
print(f"Retrieved Contexts: {len(res1.contexts)}")


## 3. Complex Query (Comparison)
Here we ask a question requiring mathematical operations or synthesizing information across multiple differing concepts. The monolith lacks calculation tools, meaning it might struggle compared to System 2.


In [ ]:
# Note: Edit the tickers below based on the filings you actually loaded
query2 = "What is the difference in total revenue between AAPL (FY2024) and MSFT (FY2024) in billions?"
print(f"Query: {query2}\n")

res2 = pipeline.query(query2)

print("\n" + "="*50)
print(f"Answer:\n{res2.answer}")
print("="*50 + "\n")

print(f"Tokens Used: {res2.metrics.token_usage.total_tokens}")
print(f"Latency: {res2.metrics.latency_seconds:.2f}s")
print(f"Retrieved Contexts: {len(res2.contexts)}")
